In [1]:
import pandas as pd
import requests as rq
import bs4
import plotly.express as px
import re
import numpy as np
import plotly.graph_objects as go

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
page = rq.get(url, headers=headers)
bs4page =  bs4.BeautifulSoup(page.text, "html.parser")
tables = bs4page.find("table", {"class": "wikitable"})

In [3]:
gdp = pd.read_html(str(tables))[0]
gdp = gdp.dropna()

C:\Users\49498\AppData\Local\Temp\ipykernel_12652\534713352.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  gdp = pd.read_html(str(tables))[0]


In [4]:
df = gdp.iloc[:, [0, 1]].copy()
df.columns = ['Area', 'IMF']

In [5]:
df = df[~df['IMF'].astype(str).str.contains('N/a|—', case=False, na=False)]
df['Area'] = df['Area'].astype(str).apply(lambda x: re.sub(r'\[.*?\]()', '', x).strip())
df['IMF'] = df['IMF'].astype(str).str.split(r'\(|\[|（').str[0]
df['IMF'] = df['IMF'].str.replace(r'[^\d]', '', regex=True)
df['IMF'] = pd.to_numeric(df['IMF'], errors='coerce')
df = df.dropna().reset_index(drop=True)
df = df[df['Area'] != 'World']

In [6]:
url = "https://en.wikipedia.org/wiki/List_of_countries_and_territories_by_the_United_Nations_geoscheme"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
page = rq.get(url, headers=headers)
bs4page =  bs4.BeautifulSoup(page.text, "html.parser")
tables = bs4page.find("table", {"class": "wikitable"})
regions = pd.read_html(str(tables))[0].iloc[:, 0].copy()
regions = regions.dropna().reset_index(drop=True)

C:\Users\49498\AppData\Local\Temp\ipykernel_12652\2750973154.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  regions = pd.read_html(str(tables))[0].iloc[:, 0].copy()


In [7]:
def getcountry(area):
    area = area.lower()
    if 'macau' in area or 'macao' in area or 'taiwan' in area: 
        return 'China'
    for name in regions:
        country_or_area = str(name)
        if area.lower() in country_or_area.lower():
            testname = re.split(r'\(|\[|（', country_or_area)[0].strip()
            if ',' in testname:
                country = testname.split(',', 1)[0].strip()
                return country
            else:
                return area
    return area

In [8]:
df['Country'] = df['Area'].apply(getcountry)
df['Country'] = df['Country'].str.title()
df = df.sort_values(by='IMF', ascending=True)

In [9]:
fig = px.bar(
    df,
    x="Country",
    y="IMF",
    color="Area",
    labels={"IMF": "GDP (million USD)", "Area": "Area", "Country": "Country"},
    barmode="stack",
)
fig.update_layout(
    showlegend=False,
    font={'family': 'Helvetica'},
    xaxis={'categoryorder': 'total ascending', 'tickfont': {'size': 8}, 'tickangle': -90, 'dtick': 1},
    yaxis={'type': 'log', 'title': 'GDP (Million USD) - Log Scale'},
    title={'text': "GDP by Country Stacked within Regions (Logarithmic Scale)", 'x': 0.5, 'y': 0.9, 'xanchor': 'center'}
)
fig.write_html("stacked_bar.html")

In [32]:
url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
multilevel_lookup = pd.read_csv(url, sep = "\t").drop(['Level5'], axis = 1)
multilevel_lookup = multilevel_lookup.rename(columns = {
    "modify"   : "roi",
    "modify.1" : "level4",
    "modify.2" : "level3",
    "modify.3" : "level2",
    "modify.4" : "level1"})
multilevel_lookup = multilevel_lookup[['roi', 'level4', 'level3', 'level2', 'level1']]

In [33]:
id = 127
subjectData = pd.read_csv("https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv")
subjectData = subjectData.loc[(subjectData.type == 1) & (subjectData.id == id)]
subjectData = subjectData[['roi', 'volume']]

subjectData = pd.merge(subjectData, multilevel_lookup, on = "roi")
subjectData = subjectData.assign(icv = "ICV")
subjectData = subjectData.assign(comp = subjectData.volume / np.sum(subjectData.volume))
subjectData = subjectData.dropna().reset_index(drop = True)

In [34]:
cols = ['level1', 'level2', 'level3', 'level4', 'roi']
for col in cols:
    subjectData[col] = col + ": " + subjectData[col].astype(str)

In [35]:
nodes = pd.unique(subjectData[cols].values.ravel('K'))
mapping = {name: i for i, name in enumerate(nodes)}

In [ ]:
level1 = [n for n in nodes if n.startswith("level1:")]
color_palette = px.colors.qualitative.Pastel
l1colors = {name: color_palette[i % len(color_palette)] for i, name in enumerate(level1)}
colors = []
for name in nodes:
    for areas in level1:
        if any((subjectData[cols] == name).any(axis=1) & (subjectData['level1'] == areas)):
            l1color = l1colors[areas]
    colors.append(l1color)

In [ ]:
sources, targets, values, linkscolor = [], [], [], []
for i in range(len(cols) - 1):
    source, target = cols[i], cols[i+1]
    total_volume = subjectData.groupby([source, target])['volume'].sum().reset_index()
    for _, row in total_volume.iterrows():
        source_index = mapping[row[source]]
        target_index = mapping[row[target]]
        sources.append(source_index)
        targets.append(target_index)
        values.append(row['volume'])
        c = colors[source_index].replace('rgb', 'rgba').replace(')', ', 0.2)')
        linkscolor.append(c)

In [ ]:
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 25,
      thickness = 10,
      label = nodes,
      color = colors,
      line = dict(color = colors, width = 0.2)
    ),
    link = dict(
        source = sources,
        target = targets,
        value = values,
        color = linkscolor)
)])
fig.update_layout(
    title_text=f"Brain Intracranial Volume Hierarchy (Subject ID: {id}, Type: 1)",
    title_x=0.5,
    title_y=0.99,
    font=dict(size=10, family="Arial"),
    height=2000,
    width=1800,
    margin=dict(l=60, r=60, t=130, b=50),
)
fig.write_html("sankey.html")